In [5]:
%pip install waybackpy
%pip install -U sec-edgar-downloader
%pip install edgartools


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 3.3 MB/s eta 0:00:01
   -------------- ------------------------- 1.0/2.8 MB 2.9 MB/s eta 0:00:01
   ---------------------- ----------------- 1.6/2.8 MB 3.0 MB/s eta 0:00:01
   ----------------------------- ---------- 2.1/2.8 MB 3.1 MB/s eta 0:00:01
   ------------------------------------- -- 2.6/2.8 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 2.7 MB/s  0:00:01

   ---------------------------- ----------- 5/7 [httpxthrottlecache]
   ---------------------------------- ----- 6/7 [edgartools]
   ---------------------------------- ----- 6/7 [edgartools]
   ---------------------------------- ----- 6/7 [edgartools]
   ---------------------------------- ----- 6/7 [edgartools]
   ---------------------------------- ----- 6/7 [edgartools]
   ---------------------------------- ----- 6/7 [edgartools]
   ---------------------------------- ----


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from pathlib import Path
import json
import logging
from datetime import datetime

from IPython.utils import io as ipy_io
from edgar import *

# Suppress edgar library warnings
logging.getLogger('edgar').setLevel(logging.ERROR)

set_identity("your.name@example.com")

# Inputs
start_year = 2017
end_year = 2025
tickers_json_path = Path("tickers.json")
output_root = Path("sec_filings")

def load_tickers(path: Path) -> list[str]:
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict):
        data = data.get("tickers") or data.get("symbols") or []
    if not isinstance(data, list):
        raise ValueError("Tickers JSON must be a list or a dict with a 'tickers' key.")
    return [str(t).strip().upper() for t in data if str(t).strip()]

def parse_year(value) -> int | None:
    if value is None:
        return None
    if isinstance(value, datetime):
        return value.year
    text = str(value).strip()
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%Y%m%d"):
        try:
            return datetime.strptime(text, fmt).year
        except ValueError:
            continue
    try:
        return int(text[:4])
    except ValueError:
        return None

def filing_year(filing) -> int | None:
    date_value = getattr(filing, "filing_date", None) or getattr(filing, "date", None)
    return parse_year(date_value)

def report_year_from_filing(filing) -> int | None:
    for attr in ("period_end", "period_end_date", "period_of_report", "report_period", "period", "fiscal_period_end"):
        year = parse_year(getattr(filing, attr, None))
        if year is not None:
            return year
    year = filing_year(filing)
    if year is None:
        return None
    return year + 1

def filing_accession(filing) -> str | None:
    for attr in ("accession_number", "accession", "accessionNo"):
        value = getattr(filing, attr, None)
        if value:
            return str(value).replace("/", "-")
    return None

tickers = load_tickers(tickers_json_path)
output_root.mkdir(parents=True, exist_ok=True)

for ticker in tickers:
    company = Company(ticker)
    filings = company.get_filings(form="10-K")
    ticker_dir = output_root / ticker
    ticker_dir.mkdir(parents=True, exist_ok=True)

    found_any = False
    found_new = False
    year_counts: dict[int, int] = {}

    for filing in filings:
        report_year = report_year_from_filing(filing)
        if report_year is None or report_year < start_year or report_year > end_year:
            continue

        # Check if file already exists for this year
        year_files = list(ticker_dir.glob(f"{ticker}_10K_{report_year}*.txt"))
        if year_files:
            found_any = True
            continue

        accession = filing_accession(filing)
        year_counts[report_year] = year_counts.get(report_year, 0) + 1
        index = year_counts[report_year]
        extra = f"_{index}" if accession is None and index > 1 else ""
        suffix = f"_{accession}" if accession else extra

        text_content = None
        try:
            with ipy_io.capture_output():
                text_content = filing.text()  # Raw text for NLP analysis
        except Exception as e:
            print(f"Error fetching 10-K text for {ticker} {report_year}: {type(e).__name__}. Writing null.")
            text_content = None

        if text_content is None or len(text_content) < 1000:
            if text_content is None:
                if "Error fetching" not in str(e) if 'e' in locals() else True:
                    print(f"Missing or short 10-K text for {ticker} {report_year}. Writing null.")
            text_content = "null"

        output_file = ticker_dir / f"{ticker}_10K_{report_year}{suffix}.txt"
        output_file.write_text(text_content, encoding="utf-8")
        found_any = True
        found_new = True

    if not found_any:
        print(f"No 10-K filings found for {ticker} from {start_year} to {end_year}.")
    elif found_new:
        print(f"Completed {ticker}.")
    else:
        print(f"Skipped {ticker} - all files already exist.")

Skipped CAT - all files already exist.
Skipped GE - all files already exist.
Skipped RTX - all files already exist.
Skipped GEV - all files already exist.
Skipped BA - all files already exist.
Skipped UNP - all files already exist.
Skipped HON - all files already exist.
Skipped ETN - all files already exist.
Skipped DE - all files already exist.
Skipped UBER - all files already exist.
Skipped LMT - all files already exist.
Skipped PH - all files already exist.
Skipped TT - all files already exist.
Skipped HWM - all files already exist.
Skipped NOC - all files already exist.
Skipped MMM - all files already exist.
Skipped UPS - all files already exist.
Skipped GD - all files already exist.
Skipped WM - all files already exist.
Skipped ADP - all files already exist.
Skipped JCI - all files already exist.
Skipped CMI - all files already exist.
Skipped EMR - all files already exist.
Skipped FDX - all files already exist.
Skipped ITW - all files already exist.
Skipped PWR - all files already